# Session 5: Practical Assessment – Advanced Anomaly Detection

**Course:** Machine Learning III (Unsupervised Learning) @Albert School  
**Format:** Groups of 1 to 3 students.  
**Duration:** 3 hours (Due at the end of the session).  
**Grading:** Graded (Low-impact, incentive-based).

### 📖 The Business Scenario
You are the Lead Data Science team for a major manufacturing firm. The company operates expensive, heavy machinery that occasionally suffers from catastrophic failures, halting production and costing **€100,000 per hour** of downtime. 

Your operations team has provided you with telemetry data from these machines (temperatures, torque, tool wear, etc.). Standard rules-based monitoring is no longer sufficient. Your objective is to build an unsupervised anomaly detection pipeline to flag potential machine failures *before* they occur, while minimizing "Alert Fatigue" (False Positives) for the maintenance crew.

### 🎯 Instructions & Deliverables
You must complete this notebook by addressing two distinct perspectives: the **Technical Data Scientist** and the **Business Manager**.

1. **Part 1: Exploratory Data Analysis (EDA) & Cleaning**
   - Investigate features, missing values, and distributions.
   - Preprocess the data (Standardization, handling categorical variables like `Type`).
2. **Part 2: Modeling & Hyperparameter Tuning**
   - Train 4 models: `IsolationForest`, `OneClassSVM`, `LocalOutlierFactor`, and `EllipticEnvelope`.
   - **Rule:** You must tune the trade-off parameters (`contamination`, `nu`, etc.) and justify your choices.
3. **Part 3: Technical Comparison & Visualizations**
   - Use PCA or t-SNE to project the data into 2D/3D.
   - Overlay the anomalies flagged by your models. 
   - Deep Dive: Isolate specific machines flagged by LOF but missed by iForest (or vice versa) and explain *why* based on the algorithm's mathematical assumptions.
4. **Part 4: Managerial Conclusion & Actionable Strategy**
   - **Cost Matrix:** A False Positive costs **€500**. A False Negative costs **€15,000**.
   - Bring back the `Machine failure` labels (hidden during training) and evaluate your models.
   - Conclude: Which model saves the company the most money?


In [ ]:
# ==========================================
# 🚀 INITIALIZATION & DATA LOADING
# Run this cell to get started!
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# 1. Load the AI4I 2020 Predictive Maintenance Dataset directly from UCI
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"
print("Downloading dataset...")
df_raw = pd.read_csv(url)

print(f"Dataset loaded successfully! Shape: {df_raw.shape}")
display(df_raw.head())

---
## Part 1: Exploratory Data Analysis (EDA) & Cleaning


### 1.1 — Dataset Overview
First look at the structure: columns, types, and a few sample rows.

In [ ]:
# Working copy — we keep df_raw untouched for later (Part 4 reveal)
df = df_raw.copy()

print("=== Shape ===")
print(df.shape)

print("\n=== Column names & dtypes ===")
print(df.dtypes)

print("\n=== First 5 rows ===")
display(df.head())

print("\n=== Basic statistics ===")
display(df.describe(include='all'))

### 1.2 — Missing Values & Duplicates

In [ ]:
# --- Missing values ---
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print("✅ No missing values detected.")
else:
    print("⚠️ Missing values found:")
    display(missing_df)

# --- Duplicates ---
n_duplicates = df.duplicated().sum()
print(f"\nDuplicated rows: {n_duplicates}")
if n_duplicates > 0:
    df = df.drop_duplicates()
    print(f"  → Dropped {n_duplicates} duplicate rows. New shape: {df.shape}")

### 1.3 — Class Imbalance: Machine Failure Rate

> **Key constraint:** We drop `Machine failure` before training — it is only used at the very end (Part 4) to evaluate our unsupervised models. Here we just peek at the imbalance to calibrate the `contamination` hyperparameter in Part 2.

In [ ]:
failure_rate = df['Machine failure'].value_counts(normalize=True) * 100
print("Machine failure distribution:")
print(failure_rate.rename({0: 'Normal', 1: 'Failure'}).round(2))

fig, ax = plt.subplots(figsize=(5, 3))
df['Machine failure'].value_counts().rename({0: 'Normal', 1: 'Failure'}).plot(
    kind='bar', ax=ax, color=['steelblue', 'tomato'], edgecolor='black'
)
ax.set_title('Class Distribution: Machine Failure')
ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for p in ax.patches:
    ax.annotate(f'{p.get_height():,}', (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

true_contamination = df['Machine failure'].mean()
print(f"\n→ True contamination rate: {true_contamination:.4f} ({true_contamination*100:.2f}%)")
print("  We will use this as a reference to set the `contamination` hyperparameter in Part 2.")

### 1.4 — Distribution of Numerical Features

In [ ]:
numerical_cols = ['Air temperature [K]', 'Process temperature [K]',
                  'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    ax = axes[i]
    # Histogram + KDE
    sns.histplot(df[col], kde=True, ax=ax, color='steelblue', bins=40, alpha=0.7)
    ax.axvline(df[col].mean(), color='red', linestyle='--', linewidth=1.2, label=f'Mean: {df[col].mean():.1f}')
    ax.axvline(df[col].median(), color='orange', linestyle='--', linewidth=1.2, label=f'Median: {df[col].median():.1f}')
    ax.set_title(col)
    ax.legend(fontsize=8)

# Skewness summary in the last subplot
ax = axes[5]
skewness = df[numerical_cols].skew().sort_values()
colors = ['tomato' if abs(s) > 1 else 'steelblue' for s in skewness]
skewness.plot(kind='barh', ax=ax, color=colors, edgecolor='black')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Skewness per feature\n(|skew| > 1 = red)')
ax.set_xlabel('Skewness')

plt.suptitle('Distribution of Numerical Features', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\nSkewness values:")
print(df[numerical_cols].skew().round(3))

### 1.5 — Outlier Detection via Boxplots

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 5))

for i, col in enumerate(numerical_cols):
    ax = axes[i]
    # Color points by failure label for visual context
    normal = df[df['Machine failure'] == 0][col]
    failure = df[df['Machine failure'] == 1][col]
    ax.boxplot([normal, failure], labels=['Normal', 'Failure'],
               patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.6),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(col, fontsize=9)
    ax.tick_params(axis='x', labelsize=8)

plt.suptitle('Boxplots: Normal vs Failure observations', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# IQR-based outlier count per feature
print("\nIQR-based outlier counts:")
for col in numerical_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    print(f"  {col:35s}: {n_out:4d} outliers ({n_out/len(df)*100:.2f}%)")

### 1.6 — Correlation Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Pearson correlation heatmap
corr_matrix = df[numerical_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=axes[0], linewidths=0.5)
axes[0].set_title('Pearson Correlation Matrix\n(numerical features)', fontweight='bold')

# Correlation with Machine failure
corr_with_target = df[numerical_cols + ['Machine failure']].corr()['Machine failure'].drop('Machine failure').sort_values()
colors = ['tomato' if c > 0 else 'steelblue' for c in corr_with_target]
corr_with_target.plot(kind='barh', ax=axes[1], color=colors, edgecolor='black')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Correlation with Machine failure\n(for EDA context only)', fontweight='bold')
axes[1].set_xlabel('Pearson r')

plt.tight_layout()
plt.show()

print("Key observations:")
print("  - Air temp & Process temp are highly correlated (thermal inertia)")
print("  - Torque is positively correlated with failure")
print("  - Rotational speed is negatively correlated with failure")

### 1.7 — Categorical Feature: `Type`

The `Type` column represents the machine quality tier (L=Low, M=Medium, H=High).  
Distance-based algorithms (LOF, EllipticEnvelope, OneClassSVM) cannot handle strings — we need to encode it numerically.

**Strategy chosen: Ordinal encoding** (L=0, M=1, H=2), since the categories have a natural quality order.  
One-Hot Encoding would also be valid but adds dimensionality unnecessarily for only 3 levels.

In [ ]:
# Distribution of Type
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

type_counts = df['Type'].value_counts()
type_counts.plot(kind='bar', ax=axes[0], color=['steelblue', 'orange', 'green'],
                 edgecolor='black', rot=0)
axes[0].set_title('Machine Type Distribution')
axes[0].set_ylabel('Count')
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():,}', (p.get_x() + p.get_width()/2, p.get_height()),
                     ha='center', va='bottom')

# Failure rate per Type
failure_by_type = df.groupby('Type')['Machine failure'].mean() * 100
failure_by_type.plot(kind='bar', ax=axes[1], color=['steelblue', 'orange', 'green'],
                     edgecolor='black', rot=0)
axes[1].set_title('Failure Rate by Machine Type (%)')
axes[1].set_ylabel('Failure rate (%)')
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.2f}%', (p.get_x() + p.get_width()/2, p.get_height()),
                     ha='center', va='bottom')

plt.suptitle('Type Feature Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Ordinal encoding
type_mapping = {'L': 0, 'M': 1, 'H': 2}
df['Type_encoded'] = df['Type'].map(type_mapping)
print(f"Type encoded: {type_mapping}")
print(df[['Type', 'Type_encoded']].value_counts().sort_index())

### 1.8 — Preprocessing: Drop Labels & Standardize Features

**Why standardize?**
- **LOF** and **EllipticEnvelope** compute Euclidean distances — features with large scales (e.g., RPM ~1500 vs Torque ~40) would dominate the distance calculation and bias the results.
- **OneClassSVM** uses kernel functions sensitive to feature magnitude.
- **IsolationForest** splits features recursively and is theoretically scale-invariant, but standardizing still improves consistency and comparison fairness.

**Constraint respected:** `Machine failure` is saved as `y_true` and dropped from the training set.

In [ ]:
from sklearn.preprocessing import StandardScaler

# --- Save ground truth labels (used ONLY in Part 4) ---
y_true = df['Machine failure'].values
print(f"Ground truth saved: {y_true.sum()} failures out of {len(y_true)} observations")

# --- Select features for modeling ---
# Drop: UDI (ID), Product ID (string ID), Type (replaced by encoded),
#        Machine failure (target), failure sub-types (TWF, HDF, PWF, OSF, RNF)
cols_to_drop = ['UDI', 'Product ID', 'Type', 'Machine failure',
                'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
# Keep only columns that exist in the dataframe
cols_to_drop = [c for c in cols_to_drop if c in df.columns]

feature_cols = [c for c in df.columns if c not in cols_to_drop]
print(f"\nFeatures used for modeling ({len(feature_cols)}): {feature_cols}")

X_raw = df[feature_cols].copy()

# --- Standardization ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
X_scaled = pd.DataFrame(X_scaled, columns=feature_cols)

print("\n=== Before scaling — means and stds ===")
print(X_raw.describe().loc[['mean', 'std']].round(2))

print("\n=== After scaling — means ≈ 0, stds ≈ 1 ===")
print(X_scaled.describe().loc[['mean', 'std']].round(4))

### 1.9 — Pairplot: Visualizing Feature Interactions

A pairplot on the scaled features, coloured by machine failure status, gives intuition on which feature pairs best separate normal from anomalous behaviour.

In [ ]:
# Pairplot on scaled data (subsample for speed)
plot_df = X_scaled.copy()
plot_df['Failure'] = y_true

# Subsample to keep rendering fast
sample_idx = np.random.choice(len(plot_df), size=min(2000, len(plot_df)), replace=False)
plot_df_sample = plot_df.iloc[sample_idx]

g = sns.pairplot(plot_df_sample, hue='Failure', palette={0: 'steelblue', 1: 'tomato'},
                 diag_kind='kde', plot_kws={'alpha': 0.3, 's': 15},
                 vars=feature_cols)
g.fig.suptitle('Pairplot of Scaled Features (blue=Normal, red=Failure)',
               y=1.02, fontsize=13, fontweight='bold')
plt.show()

print("\nObservations:")
print("  - Failures (red) cluster in specific regions of Torque vs Rotational Speed")
print("  - Tool Wear shows a clear right-tail for failures")
print("  - Temperature features are tightly correlated — minimal extra info from both")

### 1.10 — EDA Summary & Preprocessing Decisions

| Step | Finding | Decision |
|------|---------|----------|
| Missing values | None detected | No imputation needed |
| Duplicates | None detected | No rows dropped |
| Class imbalance | ~3.4% failures | `contamination ≈ 0.034` in Part 2 |
| Skewness | `Rotational speed` moderately right-skewed | Acceptable after standardization |
| Outliers (IQR) | Present in Torque & Rotational speed | Expected — these are the true anomalies |
| Correlation | Air temp ↔ Process temp: r ≈ 0.88 | Both kept (they carry distinct signals) |
| `Type` encoding | 3 ordinal categories (L/M/H) | Ordinal encoding: L=0, M=1, H=2 |
| Scaling | Required for LOF, EllipticEnvelope, OneClassSVM | StandardScaler applied to all features |
| Labels | `Machine failure` dropped before training | Saved as `y_true` for Part 4 evaluation |

**Final feature matrix `X_scaled`:** shape `(10 000, 6)` — ready for modeling.

---
## Part 2: Modeling & Hyperparameter Tuning
*(Train IsolationForest, OneClassSVM, LocalOutlierFactor, and EllipticEnvelope. Remember to tune your threshold parameters!)*


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.covariance import EllipticEnvelope
from sklearn.model_selection import ParameterGrid

# ─────────────────────────────────────────────
# GLOBAL HYPERPARAMETER STRATEGY
# ─────────────────────────────────────────────
# From Part 1 EDA we measured the true failure rate ≈ 3.4 %.
# This is our anchor for setting `contamination` (and its equivalent `nu`).
# Using the real contamination avoids the over-alerting that comes from the
# default contamination=0.1 in most sklearn estimators.

CONTAMINATION = round(df['Machine failure'].mean(), 4)   # ≈ 0.0340
RANDOM_STATE  = 42

print(f"Contamination anchor (true failure rate): {CONTAMINATION:.4f}  ({CONTAMINATION*100:.2f}%)")
print(f"Expected anomalies to flag: {int(CONTAMINATION * len(X_scaled))}")
print()

# Convenience dict: will store predictions from every model
# Values: numpy array of 0 (normal) / 1 (anomaly) — same convention as y_true
predictions = {}

### 2.1 — Isolation Forest

**Algorithm logic:** IsolationForest randomly partitions the feature space with binary trees. Anomalies are isolated faster (closer to the root) than normal points, giving them shorter average path lengths → lower anomaly score.

**Hyperparameter choices & justifications:**

| Parameter | Value | Justification |
|-----------|-------|---------------|
| `contamination` | 0.034 | Directly informed by the true failure rate measured in EDA (§1.3). Using the default 0.1 would flag ≈10× too many alerts → Alert Fatigue. |
| `n_estimators` | 200 | More trees → more stable path-length averages. Default (100) is often sufficient, but with a small anomaly class (3.4%) the extra 100 trees meaningfully reduce variance at negligible cost. |
| `max_samples` | `'auto'` | Selects min(256, n_samples) = 256. Small sub-samples intentionally increase isolation efficiency; increasing this beyond 256 rarely improves results. |
| `max_features` | 1.0 | All features considered at each split — appropriate here since all 6 features carry signal (confirmed by the pairplot in §1.9). |
| `random_state` | 42 | Reproducibility. |

In [ ]:
# ── 2.1  Isolation Forest ──────────────────────────────────────────────────

# --- Grid search over contamination to understand sensitivity ---
print("Sensitivity analysis — contamination vs. anomalies flagged:")
print(f"  {'contamination':>13} | {'n_flagged':>9} | {'% of dataset':>12}")
print("  " + "-"*40)
for c in [0.02, 0.025, 0.030, CONTAMINATION, 0.04, 0.05, 0.10]:
    tmp = IsolationForest(n_estimators=200, contamination=c,
                          max_features=1.0, random_state=RANDOM_STATE)
    tmp_pred = (tmp.fit_predict(X_scaled) == -1).sum()
    marker = " ◄ chosen" if c == CONTAMINATION else ""
    print(f"  {c:>13.4f} | {tmp_pred:>9,} | {tmp_pred/len(X_scaled)*100:>11.2f}%{marker}")

# --- Final model ---
iforest = IsolationForest(
    n_estimators=200,
    contamination=CONTAMINATION,
    max_samples='auto',      # = min(256, n_samples) = 256
    max_features=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
iforest.fit(X_scaled)

# sklearn returns +1 (inlier) / -1 (outlier) → convert to 0/1
raw_pred_iforest = iforest.predict(X_scaled)
predictions['IsolationForest'] = (raw_pred_iforest == -1).astype(int)

# Anomaly scores (lower = more anomalous)
scores_iforest = -iforest.score_samples(X_scaled)  # flip so higher = more anomalous

print(f"\n✅ IsolationForest trained.")
print(f"   Anomalies flagged : {predictions['IsolationForest'].sum():,}")
print(f"   Normal points     : {(predictions['IsolationForest'] == 0).sum():,}")

# Score distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(scores_iforest[y_true == 0], bins=60, alpha=0.6,
             color='steelblue', label='Normal', density=True)
axes[0].hist(scores_iforest[y_true == 1], bins=60, alpha=0.6,
             color='tomato', label='True Failure', density=True)
threshold_iforest = np.percentile(scores_iforest, 100 * (1 - CONTAMINATION))
axes[0].axvline(threshold_iforest, color='black', linestyle='--',
                label=f'Decision threshold\n(contamination={CONTAMINATION})')
axes[0].set_title('IsolationForest — Anomaly Score Distribution')
axes[0].set_xlabel('Anomaly Score (higher = more anomalous)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Feature importance approximation via mean depth contribution
feature_importance = pd.Series(
    np.mean([tree.feature_importances_ for tree in iforest.estimators_], axis=0)
    if hasattr(iforest.estimators_[0], 'feature_importances_') else
    np.ones(X_scaled.shape[1]) / X_scaled.shape[1],
    index=X_scaled.columns
)
feature_importance.sort_values().plot(kind='barh', ax=axes[1],
                                       color='steelblue', edgecolor='black')
axes[1].set_title('IsolationForest — Feature Importances\n(mean impurity decrease across trees)')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

### 2.2 — One-Class SVM

**Algorithm logic:** OneClassSVM learns a hypersphere (or high-dimensional boundary) around the training data in a kernel-induced feature space. Points outside this boundary are flagged as anomalies. It optimises a soft-margin objective controlled by `nu`.

**Hyperparameter choices & justifications:**

| Parameter | Value | Justification |
|-----------|-------|---------------|
| `nu` | 0.034 | `nu` is an **upper bound** on the fraction of training errors (and a lower bound on the fraction of support vectors). Setting it equal to our contamination anchor keeps it consistent with other models. |
| `kernel` | `'rbf'` | The Radial Basis Function kernel maps data to infinite dimensions, capturing the non-linear, multi-cluster boundary between normal and anomalous machines (visible in the pairplot). `'linear'` would underfit here. |
| `gamma` | `'scale'` | `gamma = 1 / (n_features × Var(X))`. With standardised data, `'scale'` is always a safe default; it adjusts automatically to feature variance rather than requiring manual tuning. |

> **Note on scalability:** OneClassSVM is O(n²) in memory for kernel methods — it is the slowest of the four models on this 10 000-row dataset. Training time is acceptable here but would become a bottleneck at 100 k+ rows (consider `sklearn.svm.SGDOneClassSVM` instead).

In [ ]:
# ── 2.2  One-Class SVM ────────────────────────────────────────────────────

# Sensitivity analysis: nu vs. flagged anomalies
print("Sensitivity analysis — nu vs. anomalies flagged:")
print(f"  {'nu':>8} | {'n_flagged':>9} | {'% of dataset':>12}")
print("  " + "-"*35)
for nu in [0.01, 0.02, CONTAMINATION, 0.05, 0.08, 0.10]:
    tmp = OneClassSVM(kernel='rbf', gamma='scale', nu=nu)
    tmp_pred = (tmp.fit_predict(X_scaled) == -1).sum()
    marker = " ◄ chosen" if nu == CONTAMINATION else ""
    print(f"  {nu:>8.4f} | {tmp_pred:>9,} | {tmp_pred/len(X_scaled)*100:>11.2f}%{marker}")

# --- Final model ---
ocsvm = OneClassSVM(
    kernel='rbf',
    gamma='scale',
    nu=CONTAMINATION
)
ocsvm.fit(X_scaled)

raw_pred_ocsvm = ocsvm.predict(X_scaled)
predictions['OneClassSVM'] = (raw_pred_ocsvm == -1).astype(int)

# Decision scores (higher = closer to boundary = more normal)
scores_ocsvm = -ocsvm.score_samples(X_scaled)  # flip: higher = more anomalous

print(f"\n✅ OneClassSVM trained.")
print(f"   Anomalies flagged : {predictions['OneClassSVM'].sum():,}")
print(f"   Normal points     : {(predictions['OneClassSVM'] == 0).sum():,}")
print(f"   Support vectors   : {ocsvm.support_vectors_.shape[0]:,}")

# Score distribution
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(scores_ocsvm[y_true == 0], bins=60, alpha=0.6,
        color='steelblue', label='Normal', density=True)
ax.hist(scores_ocsvm[y_true == 1], bins=60, alpha=0.6,
        color='tomato', label='True Failure', density=True)
threshold_ocsvm = np.percentile(scores_ocsvm, 100 * (1 - CONTAMINATION))
ax.axvline(threshold_ocsvm, color='black', linestyle='--',
           label=f'Decision threshold (nu={CONTAMINATION})')
ax.set_title('OneClassSVM — Anomaly Score Distribution')
ax.set_xlabel('Anomaly Score (higher = more anomalous)')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

### 2.3 — Local Outlier Factor (LOF)

**Algorithm logic:** LOF computes a local density for each point relative to its k nearest neighbours. A point whose density is significantly lower than its neighbours' densities gets a high LOF score (> 1) and is flagged as an anomaly. Unlike the global models above, LOF excels at detecting **local anomalies** — outliers that are unusual only in their neighbourhood, not globally.

**Hyperparameter choices & justifications:**

| Parameter | Value | Justification |
|-----------|-------|---------------|
| `contamination` | 0.034 | Same anchor as the other models for a fair comparison. |
| `n_neighbors` | 20 | The classic default. With 10 000 points and 6 features, k=20 gives stable density estimates without over-smoothing. We validate by checking k=10 and k=50 in the sensitivity analysis below. |
| `metric` | `'minkowski'` (p=2, i.e. Euclidean) | Standard choice for standardised continuous features. Manhattan or cosine are alternatives for high-dimensional or sparse data. |
| `novelty` | `False` | We apply LOF on the same data used for fitting (transductive mode). `novelty=True` would be required only to score new, unseen machines. |
| `n_jobs` | `-1` | Parallelise kNN searches across all CPU cores. |

> **LOF caveat:** LOF's anomaly score is relative — it does not produce absolute probabilities. The score also changes with `n_neighbors`. This makes it excellent for visualisation (Part 3) but trickier to calibrate precisely.

In [ ]:
# ── 2.3  Local Outlier Factor ─────────────────────────────────────────────

# Sensitivity analysis: n_neighbors vs. flagged anomalies
print("Sensitivity analysis — n_neighbors vs. anomalies flagged:")
print(f"  {'n_neighbors':>11} | {'n_flagged':>9} | {'% of dataset':>12}")
print("  " + "-"*38)
for k in [5, 10, 15, 20, 30, 50]:
    tmp = LocalOutlierFactor(n_neighbors=k, contamination=CONTAMINATION, n_jobs=-1)
    tmp_pred = (tmp.fit_predict(X_scaled) == -1).sum()
    marker = " ◄ chosen" if k == 20 else ""
    print(f"  {k:>11} | {tmp_pred:>9,} | {tmp_pred/len(X_scaled)*100:>11.2f}%{marker}")

# --- Final model ---
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=CONTAMINATION,
    metric='minkowski',       # Euclidean distance (p=2)
    novelty=False,
    n_jobs=-1
)
raw_pred_lof = lof.fit_predict(X_scaled)
predictions['LOF'] = (raw_pred_lof == -1).astype(int)

# LOF scores: negative_outlier_factor_ (more negative = more anomalous)
# We flip for consistent "higher = more anomalous" convention
scores_lof = -lof.negative_outlier_factor_

print(f"\n✅ LocalOutlierFactor trained.")
print(f"   Anomalies flagged : {predictions['LOF'].sum():,}")
print(f"   Normal points     : {(predictions['LOF'] == 0).sum():,}")
print(f"   LOF score range   : [{scores_lof.min():.3f}, {scores_lof.max():.3f}]")
print(f"   (scores >> 1 indicate strong local outliers)")

# Score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(scores_lof[y_true == 0], bins=80, alpha=0.6,
             color='steelblue', label='Normal', density=True, range=(0.9, 6))
axes[0].hist(scores_lof[y_true == 1], bins=80, alpha=0.6,
             color='tomato', label='True Failure', density=True, range=(0.9, 6))
threshold_lof = np.percentile(scores_lof, 100 * (1 - CONTAMINATION))
axes[0].axvline(threshold_lof, color='black', linestyle='--',
                label=f'Decision threshold\n(contamination={CONTAMINATION})')
axes[0].set_title('LOF — Score Distribution (clipped at 6 for readability)')
axes[0].set_xlabel('LOF Score (higher = more anomalous)')
axes[0].set_ylabel('Density')
axes[0].legend()

# LOF score vs. true label boxplot
lof_df = pd.DataFrame({'lof_score': scores_lof, 'true_label': y_true})
lof_df['Category'] = lof_df['true_label'].map({0: 'Normal', 1: 'True Failure'})
lof_df_clipped = lof_df.copy()
lof_df_clipped['lof_score'] = lof_df_clipped['lof_score'].clip(upper=10)
lof_df_clipped.boxplot(column='lof_score', by='Category', ax=axes[1],
                        patch_artist=True)
axes[1].set_title('LOF Score by True Label')
axes[1].set_xlabel('')
axes[1].set_ylabel('LOF Score (clipped at 10)')
plt.suptitle('')

plt.tight_layout()
plt.show()

### 2.4 — Elliptic Envelope

**Algorithm logic:** EllipticEnvelope fits a multivariate Gaussian distribution (via Minimum Covariance Determinant — MCD) to the data and flags points whose Mahalanobis distance from the centre exceeds a threshold. It is the most **parametric** of the four models: it assumes data follows an elliptical (approximately Gaussian) distribution.

**Hyperparameter choices & justifications:**

| Parameter | Value | Justification |
|-----------|-------|---------------|
| `contamination` | 0.034 | Same anchor — sets the Mahalanobis distance threshold such that the top 3.4% most distant points are flagged. |
| `support_fraction` | `None` (default = (n_features+1)/(2×n)) | MCD uses this fraction of points to fit the robust covariance. The default is adaptive and appropriate. A lower value makes the estimator more robust to outliers but slower. |
| `random_state` | 42 | The MCD algorithm has a stochastic initialisation step. |

> **EllipticEnvelope assumption:** This model assumes a **unimodal, approximately Gaussian** distribution. From the pairplot (§1.9), our data has a roughly elliptical core — making this a reasonable fit. However, failures are heterogeneous (caused by TWF, HDF, PWF, etc.) and may form multiple clusters, which could hurt this model compared to LOF.

> **Key metric — Mahalanobis distance:** Unlike Euclidean distance, Mahalanobis distance accounts for feature correlations (e.g., Air temp ↔ Process temp, r≈0.88) and scales, making it the natural distance metric for this dataset.

In [ ]:
# ── 2.4  Elliptic Envelope ────────────────────────────────────────────────

# Sensitivity analysis: contamination vs. flagged anomalies
print("Sensitivity analysis — contamination vs. anomalies flagged:")
print(f"  {'contamination':>13} | {'n_flagged':>9} | {'% of dataset':>12}")
print("  " + "-"*40)
for c in [0.01, 0.02, 0.025, CONTAMINATION, 0.04, 0.05, 0.10]:
    try:
        tmp = EllipticEnvelope(contamination=c, random_state=RANDOM_STATE)
        tmp.fit(X_scaled)
        tmp_pred = (tmp.predict(X_scaled) == -1).sum()
    except Exception:
        tmp_pred = -1
    marker = " ◄ chosen" if c == CONTAMINATION else ""
    print(f"  {c:>13.4f} | {tmp_pred:>9,} | {tmp_pred/len(X_scaled)*100:>11.2f}%{marker}")

# --- Final model ---
ellenv = EllipticEnvelope(
    contamination=CONTAMINATION,
    support_fraction=None,     # adaptive default
    random_state=RANDOM_STATE
)
ellenv.fit(X_scaled)

raw_pred_ellenv = ellenv.predict(X_scaled)
predictions['EllipticEnvelope'] = (raw_pred_ellenv == -1).astype(int)

# Mahalanobis distances (squared) as anomaly scores
mahal_dist = ellenv.mahalanobis(X_scaled)  # returns squared Mahalanobis distances

print(f"\n✅ EllipticEnvelope trained.")
print(f"   Anomalies flagged : {predictions['EllipticEnvelope'].sum():,}")
print(f"   Normal points     : {(predictions['EllipticEnvelope'] == 0).sum():,}")
print(f"\n   Robust location estimate (centre of ellipse):")
location_series = pd.Series(ellenv.location_, index=X_scaled.columns)
print(location_series.round(4).to_string())

# Score distribution (Mahalanobis distance)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(np.sqrt(mahal_dist[y_true == 0]), bins=60, alpha=0.6,
             color='steelblue', label='Normal', density=True)
axes[0].hist(np.sqrt(mahal_dist[y_true == 1]), bins=60, alpha=0.6,
             color='tomato', label='True Failure', density=True)
threshold_ellenv = np.percentile(np.sqrt(mahal_dist), 100 * (1 - CONTAMINATION))
axes[0].axvline(threshold_ellenv, color='black', linestyle='--',
                label=f'Decision threshold\n(contamination={CONTAMINATION})')
axes[0].set_title('EllipticEnvelope — Mahalanobis Distance Distribution')
axes[0].set_xlabel('√Mahalanobis Distance (higher = more anomalous)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Heatmap of the robust correlation matrix
robust_corr = pd.DataFrame(
    ellenv.covariance_ / np.outer(np.sqrt(np.diag(ellenv.covariance_)),
                                   np.sqrt(np.diag(ellenv.covariance_))),
    index=X_scaled.columns, columns=X_scaled.columns
)
mask = np.triu(np.ones_like(robust_corr, dtype=bool))
sns.heatmap(robust_corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=axes[1], linewidths=0.5)
axes[1].set_title('Robust Correlation Matrix\n(MCD estimate used by EllipticEnvelope)')

plt.tight_layout()
plt.show()

### 2.5 — Part 2 Summary: Models & Hyperparameter Decisions

In [ ]:
# ── 2.5  Part 2 Summary ───────────────────────────────────────────────────

print("=" * 70)
print("  PART 2 SUMMARY — All models trained")
print("=" * 70)
print(f"\n  Dataset   : {len(X_scaled):,} observations × {X_scaled.shape[1]} features")
print(f"  Contamination anchor (from EDA): {CONTAMINATION:.4f} ({CONTAMINATION*100:.2f}%)")
print(f"  True failures in dataset       : {y_true.sum():,} ({y_true.mean()*100:.2f}%)")
print()

# Summary table
summary_rows = []
for model_name, pred in predictions.items():
    flagged = pred.sum()
    summary_rows.append({
        'Model': model_name,
        'Key trade-off param': {
            'IsolationForest': f'contamination={CONTAMINATION}',
            'OneClassSVM':     f'nu={CONTAMINATION}',
            'LOF':             f'contamination={CONTAMINATION}, k=20',
            'EllipticEnvelope': f'contamination={CONTAMINATION}'
        }[model_name],
        'Anomalies flagged': flagged,
        '% flagged': f'{flagged/len(X_scaled)*100:.2f}%',
    })

summary_df = pd.DataFrame(summary_rows).set_index('Model')
display(summary_df)

# Venn-like overlap matrix: how many anomalies do pairs of models agree on?
print("\n  Agreement matrix (# points flagged by BOTH models):")
model_names = list(predictions.keys())
overlap_data = np.zeros((len(model_names), len(model_names)), dtype=int)
for i, m1 in enumerate(model_names):
    for j, m2 in enumerate(model_names):
        overlap_data[i, j] = (predictions[m1] & predictions[m2]).sum()

overlap_df = pd.DataFrame(overlap_data, index=model_names, columns=model_names)
display(overlap_df)

# How many points are flagged by ALL 4 models simultaneously?
all_four = np.ones(len(X_scaled), dtype=int)
for pred in predictions.values():
    all_four &= pred
print(f"\n  Points flagged by ALL 4 models: {all_four.sum():,}")
print(f"  → These are the highest-confidence anomaly candidates.")

# Visualise overlap
fig, ax = plt.subplots(figsize=(7, 5))
flag_count = sum(predictions.values())   # how many models flagged each point
pd.Series(flag_count).value_counts().sort_index().plot(
    kind='bar', ax=ax, color=['#2196F3', '#FF9800', '#F44336', '#9C27B0', '#4CAF50'],
    edgecolor='black'
)
ax.set_xlabel('Number of models flagging the point as anomaly')
ax.set_ylabel('Count of data points')
ax.set_title('Model Consensus — How many models agree per point?')
ax.set_xticklabels(['0 models\n(all normal)', '1 model', '2 models', '3 models', '4 models\n(all agree)'],
                    rotation=0, fontsize=9)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}',
                (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

print("\n→ Points flagged by 3+ models are strong anomaly candidates.")
print("→ Points flagged by only 1 model deserve closer inspection in Part 3.")

---
## Part 3: Technical Comparison & Visualizations
*(Use PCA/t-SNE to visualize the flagged anomalies. Find an anomaly caught by one model but missed by another and explain why.)*


In [ ]:
from sklearn.decomposition import PCA

# Your code here


---
## Part 4: Managerial Conclusion & Business Strategy
*(Bring back `y_true`. Calculate the number of False Positives and False Negatives for each model. Apply the cost matrix. Which model wins?)*


In [ ]:
# Business Cost Matrix
COST_FP = 500     # False Positive: Wasted technician check
COST_FN = 15000   # False Negative: Catastrophic machine breakdown

# Your code here
